# 5. Register and verify the candidate

Register the already saved winner, assign a movable alias, and verify the model can be reloaded. No deployment occurs.

## Execution parameters
Airflow injects these values into the tagged parameter cell. For manual execution, keep the defaults and run the five notebooks in order.

In [ ]:
# Notebook inputs.
# Airflow supplies the paths and run ID; manual execution uses these defaults.
project_root = ""
run_dir = ""
workflow_run_id = ""
experiment_name = "iris-pipeline-comparison"
registered_model_name = "iris-pipeline-classifier"

## Locate files and connect to MLflow
SQLite holds experiment and registry records. This run folder holds data and small result files shared between notebook tasks.

In [ ]:
import json
from pathlib import Path

import mlflow
import pandas as pd
from mlflow import MlflowClient

# Find the project folder.
# Looking for requirements.txt lets this notebook start from a subfolder too.
if project_root:
    project_path = Path(project_root).resolve()
else:
    project_path = Path.cwd().resolve()
while not (project_path / "requirements.txt").is_file():
    if project_path == project_path.parent:
        raise RuntimeError("Open the notebook from inside the workshop project.")
    project_path = project_path.parent

# Choose the output folder.
# Separate Airflow run folders prevent one execution from overwriting another.
if run_dir:
    run_path = Path(run_dir)
else:
    run_path = project_path / "outputs" / "manual"
run_path.mkdir(parents=True, exist_ok=True)

# Connect to MLflow.
# One database keeps both models, their runs, and registered versions together.
tracking_uri = "sqlite:///" + (project_path / "mlflow.db").as_posix()
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)
client = MlflowClient()

print("Run folder:", run_path)

## Register the winner
The registered model name groups numbered versions. Each registration adds a version linked to a saved model and source run.

In [ ]:
winner = json.loads((run_path / "winner.json").read_text(encoding="utf-8"))
split = json.loads((run_path / "split.json").read_text(encoding="utf-8"))
assert winner["execution_id"] == split["execution_id"], "Compare the current execution before registering."

# Register the saved winner.
# This creates a numbered version without fitting the model again.
model_version = mlflow.register_model(winner["model_uri"], registered_model_name)
print("Registered model:", model_version.name)
print("Version:", model_version.version)

## Assign candidate
An alias points to a model version. Moving it preserves older versions; it does not deploy the model.

In [ ]:
# Select the candidate version.
# The alias is a stable name that can point to a different version after a later run.
client.set_registered_model_alias(
    name=registered_model_name,
    alias="candidate",
    version=model_version.version,
)
candidate = client.get_model_version_by_alias(registered_model_name, "candidate")
print("Candidate version:", candidate.version)

## Reload, predict, and trace verification
A trace records an instrumented operation; a span is one timed step inside it. We create a verify_candidate trace with two children: load_candidate and predict_candidate. It records model references, prediction inputs/outputs, timing, and errors, and links to the winning training run. This is separate from numeric training metrics.

Loading by alias retrieves the saved winner. The original source-run and accuracy checks still verify it. If a traced operation raises an exception, MLflow records its error and Airflow fails the task.

In [ ]:
import mlflow.sklearn
from sklearn.metrics import accuracy_score

candidate_uri = f"models:/{registered_model_name}@candidate"
X_test = pd.DataFrame(split["X_test"], columns=split["feature_names"], dtype="float64")

# Choose where to store the trace.
# This notebook has a separate Python process, so it selects the experiment
# explicitly.
mlflow.set_experiment(experiment_name)

# Trace candidate verification.
# The root span links to the winning run; nested spans time loading and prediction.
with mlflow.start_span(
    name="verify_candidate",
    run_id=winner["run_id"],
) as verification:
    verification.set_inputs(
        {
            "model_uri": candidate_uri,
            "version": str(candidate.version),
        },
    )
    mlflow.update_current_trace(tags={"workflow_run_id": split["execution_id"]})

    with mlflow.start_span(name="load_candidate") as loading:
        loading.set_inputs({"model_uri": candidate_uri})
        candidate_model = mlflow.sklearn.load_model(candidate_uri)
        # Record which model was loaded.
        # The trace needs its version and source run; the model itself is already a
        # saved artifact.
        loading.set_outputs({
            "model_type": winner["model_type"],
            "version": str(candidate.version),
            "source_run_id": candidate.run_id,
        })

    with mlflow.start_span(name="predict_candidate") as predicting:
        # Record the prediction inputs and outputs.
        # The small public Iris batch can be inspected directly in the trace viewer.
        predicting.set_inputs({"rows": X_test.to_dict(orient="records")})
        candidate_predictions = candidate_model.predict(X_test)
        predicting.set_outputs({"predictions": candidate_predictions.tolist()})

    candidate_accuracy = accuracy_score(split["y_test"], candidate_predictions)
    assert candidate.run_id == winner["run_id"]
    assert candidate_accuracy == winner["accuracy"]
    verification.set_outputs({
        "accuracy": candidate_accuracy,
        "verified": True,
    })

trace_id = verification.trace_id
# Finish saving the trace.
# Flush pending writes so the completed Airflow task leaves its trace available in
# MLflow.
mlflow.flush_trace_async_logging()

## Save the verification record
Keep the candidate version, source run, accuracy, and trace ID together for this execution.

In [ ]:
summary = {
    "execution_id": split["execution_id"],
    "registered_model": registered_model_name,
    "version": str(candidate.version),
    "alias": "candidate",
    "run_id": candidate.run_id,
    "accuracy": candidate_accuracy,
    "trace_id": trace_id,
}
(run_path / "candidate.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)
print("Verified candidate accuracy:", candidate_accuracy)
print("Summary:", run_path / "candidate.json")
print("Verification trace ID:", trace_id)